## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2023-01-17 01:39:27--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2023-01-17 01:39:28 (29.0 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [ ]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [ ]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [ ]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


4.65630578994751


In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


oTo.JUZ!!zqe!
xBP qbs$Gy'AcOmrLwwt
p$x;Seh-onQbfM?OjKbn'NwUAW -Np3fkz$FVwAUEa-wzWC -wQo-R!v -Mj?,SPiTyZ;o-opr$mOiPJEYD-CfigkzD3p3?zvS;ADz;.y?o,ivCuC'zqHxcVT cHA
rT'Fd,SBMZyOslg!NXeF$sBe,juUzLq?w-wzP-h
ERjjxlgJzPbHxf$ q,q,KCDCU fqBOQT
SV&CW:xSVwZv'DG'NSPypDhKStKzC -$hslxIVzoivnp ,ethA:NCCGoi
tN!ljjP3fwJMwNelgUzzPGJlgihJ!d?q.d
pSPYgCuCJrIFtb
jQXg
pA.P LP,SPJi
DBcuBM:CixjJ$Jzkq,OLf3KLQLMGph$O 3DfiPHnXKuHMlyjxEiyZib3FaHV-oJa!zoc'XSP :CKGUhd?lgCOF$;;DTHZMlvvcmZAm;:iv'MMgO&Ywbc;BLCUd&vZINLIzkuTGZa
D.?


## The mathematical trick in self-attention

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [ ]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

True

In [ ]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


True

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [ ]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [ ]:
k.var()

tensor(1.0449)

In [ ]:
q.var()

tensor(1.0700)

In [ ]:
wei.var()

tensor(1.0918)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [ ]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [ ]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [ ]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



### Full finished code, for reference

You may want to refer directly to the git repo instead though.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5090, val loss 2.5058
step 300: train loss 2.4198, val loss 2.4340
step 400: train loss 2.3503, val loss 2.3567
step 500: train loss 2.2970, val loss 2.3136
step 600: train loss 2.2410, val loss 2.2506
step 700: train loss 2.2062, val loss 2.2198
step 800: train loss 2.1638, val loss 2.1871
step 900: train loss 2.1232, val loss 2.1494
step 1000: train loss 2.1020, val loss 2.1293
step 1100: train loss 2.0704, val loss 2.1196
step 1200: train loss 2.0382, val loss 2.0798
step 1300: train loss 2.0249, val loss 2.0640
step 1400: train loss 1.9922, val loss 2.0354
step 1500: train loss 1.9707, val loss 2.0308
step 1600: train loss 1.9614, val loss 2.0474
step 1700: train loss 1.9393, val loss 2.0130
step 1800: train loss 1.9070, val loss 1.9943
step 1900: train loss 1.9057, val loss 1.9871
step 2000: train loss 1.8834, val loss 1.9954
step 2100: train loss 1.

## FlowScope: watch the numbers flow forward and backward

Run the next cell once (it writes the `flowscope` package), then run the GPT cell below it. The picture redraws every 50 steps while training runs.

- **Top lane**: the residual stream going forward. Color = RMS of the activations. Red means the signal is shrinking toward zero.
- **Bottom lane**: the gradient `dloss/dx` flowing back from the loss, shown relative to the gradient at the last block. Dark red toward the left = **blockage**: the early layers get almost no learning signal.
- **Pulses** sweep forward, then backward. Their size is the size of the signal; watch them shrink when something blocks.
- **Strips**: the actual 64 numbers of one token's residual vector (left) and its gradient (right) at each point.
- **Per block**: attention pattern of head 1, `p̂` = average max attention prob (near 1 = saturated softmax), MLP dead units, update/data ratio.
- **Bottom-left chart**: gradient reaching each depth. Every run you do is kept as a dashed line, so you can compare.
- **Representation tour** (bottom): every token's residual vector at each block, seen through one slowly rotating 3D window (a grand tour through the top principal directions). `eff.dim` = how many directions the cloud really uses; without residuals it collapses toward 1-2 already at step 0. Click a panel to enlarge.


In [ ]:
# Writes the flowscope package next to this notebook (generated by scripts/build_notebook.py)
import os, sys, importlib
files = {
"flowscope/__init__.py": "from flowscope.scope import FlowScope\n\n__all__ = [\"FlowScope\"]\n",
"flowscope/scope.py": "\"\"\"FlowScope: live view of the forward and backward pass through a residual-stream transformer.\n\nAttach once, then train as usual; it hooks the model and optimizer and redraws every `every` steps.\n\n    scope = FlowScope(model, optimizer, every=50, label=\"baseline\")\n    ... your normal training loop ...\n    scope.close()   # optional: remove hooks\n\nIn a notebook it draws inline; in a plain script it serves a live page at http://127.0.0.1:8765\n(or run an unmodified script with `flowscope run train.py`).\n\nTop lane: the residual stream going forward (color = RMS of the activations).\nBottom lane: dloss/dx flowing back (color = size relative to the gradient at the last block).\nA dark red bottom lane toward the left is a blockage: early layers get almost no learning signal.\n\"\"\"\nimport base64, json, math, os, struct, zlib\nimport torch\nimport torch.nn as nn\n\nfrom flowscope.tour import Tour, embed, rows, sample\n\n# ---------------------------------------------------------------- colors / images\n\ndef _lerp(stops, t):\n    t = min(max(t, 0.0), 1.0) * (len(stops) - 1)\n    i = min(int(t), len(stops) - 2)\n    f = t - i\n    a, b = stops[i], stops[i + 1]\n    return tuple(int(a[k] + (b[k] - a[k]) * f) for k in range(3))\n\n_DIV = [(56, 132, 255), (20, 24, 36), (255, 128, 32)]                       # -1 .. 0 .. +1\n_SEQ = [(12, 14, 24), (88, 40, 140), (220, 80, 90), (252, 210, 60)]         # attention 0 .. 1\n_HEALTH = [(90, 12, 20), (170, 40, 40), (120, 120, 130), (40, 210, 235), (240, 80, 200)]\n\ndef _health(log_ratio):\n    \"\"\"log10(value/reference): -8 blocked (dark red), 0 healthy (cyan), +3 exploding (magenta).\"\"\"\n    t = (log_ratio + 8) / 8 * 0.75 if log_ratio <= 0 else 0.75 + min(log_ratio, 3) / 3 * 0.25\n    return \"rgb(%d,%d,%d)\" % _lerp(_HEALTH, t)\n\ndef _fwd_color(rms):\n    \"\"\"Forward activations: shrinking toward 0 is the problem, so penalize small RMS harder than large.\"\"\"\n    l = _log(rms)\n    return _health(3 * l if l < 0 else 0.7 * l)\n\ndef _png(pixels, w, h):\n    raw = b\"\".join(b\"\\x00\" + bytes(c for px in pixels[r * w:(r + 1) * w] for c in px) for r in range(h))\n    def chunk(tag, data):\n        return struct.pack(\">I\", len(data)) + tag + data + struct.pack(\">I\", zlib.crc32(tag + data) & 0xFFFFFFFF)\n    png = b\"\\x89PNG\\r\\n\\x1a\\n\" + chunk(b\"IHDR\", struct.pack(\">IIBBBBB\", w, h, 8, 2, 0, 0, 0)) \\\n        + chunk(b\"IDAT\", zlib.compress(raw)) + chunk(b\"IEND\", b\"\")\n    return \"data:image/png;base64,\" + base64.b64encode(png).decode()\n\ndef _strip(vec):\n    \"\"\"A vertical strip: one cell per channel, diverging colors, normalized to its own max.\"\"\"\n    m = max((abs(v) for v in vec), default=0) or 1.0\n    return _png([_lerp(_DIV, 0.5 + 0.5 * v / m) for v in vec], 1, len(vec))\n\ndef _heat(mat):\n    h, w = len(mat), len(mat[0])\n    return _png([_lerp(_SEQ, v) for row in mat for v in row], w, h)\n\ndef _sci(x):\n    if x is None or x != x:\n        return \"-\"\n    if x == 0:\n        return \"0\"\n    e = int(math.floor(math.log10(abs(x))))\n    return f\"{x:.2f}\" if -2 <= e <= 2 else f\"{x / 10 ** e:.1f}e{e}\"\n\ndef _log(x, floor=1e-30):\n    return math.log10(max(x, floor))\n\n# ---------------------------------------------------------------- model discovery\n\ndef _child(mod, *names):\n    for n in names:\n        if hasattr(mod, n) and isinstance(getattr(mod, n), nn.Module):\n            return getattr(mod, n)\n    return None\n\ndef _in_notebook():\n    try:\n        import sys\n        from IPython import get_ipython\n        # Jupyter / Colab kernels; plain python and the IPython terminal have no kernel\n        return \"google.colab\" in sys.modules or hasattr(get_ipython(), \"kernel\")\n    except Exception:\n        return False\n\ndef _find_blocks(model):\n    for path in (\"blocks\", \"transformer.h\", \"layers\", \"h\"):\n        m = model\n        for part in path.split(\".\"):\n            m = getattr(m, part, None)\n        if isinstance(m, (nn.Sequential, nn.ModuleList)) and len(m):\n            return list(m)\n    raise ValueError(\"FlowScope: couldn't find the transformer blocks (looked for model.blocks / model.transformer.h)\")\n\n# ---------------------------------------------------------------- the scope\n\nclass FlowScope:\n    runs = {}  # label -> latest gradient profile, shared across scopes so runs can be compared\n\n    def __init__(self, model, optimizer, every=50, label=None, token=-1, sink=\"auto\", port=8765,\n                 runs_file=\".flowscope/runs.json\", tour_points=256, record=None, host=\"127.0.0.1\"):\n        old = getattr(model, \"_flowscope\", None)\n        if old is not None:\n            old.close()\n        model._flowscope = self\n        self.model, self.opt, self.every, self.token = model, optimizer, every, token\n        self.label = label or f\"run {len(FlowScope.runs) + 1}\"\n        self.blocks = _find_blocks(model)\n        self.L = len(self.blocks)\n        self.steps, self.active, self.cur, self.hist, self._hooks, self._snap = 0, False, None, [], [], {}\n        self.tour_points, self._tour, self.tour_payload = tour_points, Tour() if tour_points else None, None\n        self.record = record  # JSONL file: one tour frame per redraw, for replay (e.g. in GrandTourVision)\n        self._install()\n        self._handle = self._server = None\n        waiting = self._frame('<div style=\"padding:24px;color:#8b93a7\">FlowScope attached. Waiting for the first training step...</div>')\n        if sink == \"auto\":\n            sink = \"notebook\" if _in_notebook() else \"server\"\n        if sink == \"notebook\":\n            from IPython.display import HTML, display\n            self._HTML = HTML\n            self._handle = display(HTML(waiting), display_id=True)\n        elif sink == \"server\" or hasattr(sink, \"publish\"):\n            if sink == \"server\":\n                from flowscope.server import LiveServer\n                sink = LiveServer.shared(port, host)\n            self._server = sink\n            self._server.publish(waiting)\n            self.runs_file = runs_file\n            self._load_runs()\n\n    # ---- hooks -------------------------------------------------------------------\n\n    def _install(self):\n        H = self._hooks\n        H.append(self.model.register_forward_pre_hook(self._on_model_pre))\n        H.append(self.model.register_forward_hook(self._on_model_out))\n        for i, b in enumerate(self.blocks):\n            if i == 0:\n                H.append(b.register_forward_pre_hook(lambda m, a: self._cap((\"s\", 0), a[0])))\n            H.append(b.register_forward_hook(lambda m, a, o, i=i: self._cap((\"s\", 2 * i + 2), o)))\n            ln2 = _child(b, \"ln2\", \"ln_2\", \"post_attention_layernorm\")\n            if ln2 is not None:\n                H.append(ln2.register_forward_pre_hook(lambda m, a, i=i: self._cap((\"s\", 2 * i + 1), a[0])))\n            sa, ff = _child(b, \"sa\", \"attn\", \"self_attn\", \"attention\"), _child(b, \"ffwd\", \"mlp\", \"feed_forward\")\n            if sa is not None:\n                H.append(sa.register_forward_hook(lambda m, a, o, i=i: self._cap((\"a\", i), o, grad=False)))\n                heads = [h for h in sa.modules() if hasattr(h, \"tril\") and isinstance(getattr(h, \"dropout\", None), nn.Module)]\n                for k, h in enumerate(heads):  # attention probs are the input to each head's dropout\n                    H.append(h.dropout.register_forward_pre_hook(lambda m, a, i=i, k=k: self._cap_attn(i, k, a[0])))\n            if ff is not None:\n                H.append(ff.register_forward_hook(lambda m, a, o, i=i: self._cap((\"f\", i), o, grad=False)))\n                for act in ff.modules():\n                    if isinstance(act, (nn.ReLU, nn.GELU, nn.SiLU)):\n                        H.append(act.register_forward_hook(lambda m, a, o, i=i: self._cap_relu(i, o)))\n                        break\n        head = _child(self.model, \"lm_head\", \"head\")\n        if head is not None:\n            H.append(head.register_forward_hook(lambda m, a, o: self._cap((\"logits\", 0), o)))\n        H.append(self.opt.register_step_pre_hook(self._on_opt_pre))\n        H.append(self.opt.register_step_post_hook(self._on_opt_post))\n        # parameter groups for grad size and update/data ratio\n        self._groups = [(\"emb\", [p for n, p in self.model.named_parameters() if \"embed\" in n or n.startswith((\"wte\", \"wpe\"))])]\n        for i, b in enumerate(self.blocks):\n            sa, ff = _child(b, \"sa\", \"attn\", \"self_attn\", \"attention\"), _child(b, \"ffwd\", \"mlp\", \"feed_forward\")\n            self._groups.append(((\"a\", i), list(sa.parameters()) if sa is not None else []))\n            self._groups.append(((\"f\", i), list(ff.parameters()) if ff is not None else []))\n        if head is not None:\n            self._groups.append((\"head\", list(head.parameters())))\n\n    def close(self):\n        for h in self._hooks:\n            h.remove()\n        self._hooks = []\n        if getattr(self.model, \"_flowscope\", None) is self:\n            del self.model._flowscope\n\n    def _training(self):\n        return self.model.training and torch.is_grad_enabled()\n\n    def _on_model_pre(self, m, args):\n        self.active = self._training() and self.steps % self.every == 0\n        if self.active:\n            self.cur = dict(fwd={}, grad={}, vec={}, gvec={}, attn={}, relu={}, cloud={}, tokens=None, loss=None)\n            idx = args[0] if args else None\n            if self._tour and torch.is_tensor(idx) and idx.dim() == 2 and not idx.is_floating_point():\n                self.cur[\"tokens\"] = idx[:rows(*idx.shape, self.tour_points)].reshape(-1).tolist()\n\n    def _on_model_out(self, m, args, out):\n        if self.active and isinstance(out, tuple) and len(out) == 2 and torch.is_tensor(out[1]):\n            self.cur[\"loss\"] = out[1].item()\n        elif self.active:\n            self.active = False  # e.g. generate(): no targets, no backward coming\n\n    def _pick(self, t):\n        return t.detach()[0, self.token].float().cpu().tolist() if t.dim() == 3 else []\n\n    def _cap(self, key, t, grad=True):\n        if not (self.active and torch.is_tensor(t)):\n            return\n        c = self.cur\n        c[\"fwd\"][key] = t.detach().float().pow(2).mean().sqrt().item()\n        if self._tour and key[0] == \"s\" and key[1] % 2 == 0 and t.dim() == 3:  # block boundaries only\n            c[\"cloud\"][key[1] // 2] = sample(t, self.tour_points)\n            c[\"T\"] = t.shape[1]\n        c[\"vec\"][key] = self._pick(t)\n        if grad and t.requires_grad:\n            def on_grad(g, key=key):\n                c[\"grad\"][key] = g.detach().float().pow(2).mean().sqrt().item()\n                c[\"gvec\"][key] = self._pick(g)\n            t.register_hook(on_grad)\n\n    def _cap_attn(self, i, k, wei):\n        if not self.active or wei.dim() != 3:\n            return\n        w = wei.detach().float()\n        maxp = w.max(-1).values.mean().item()\n        ent = -(w * torch.log(w.clamp_min(1e-12))).sum(-1).mean().item()\n        d = self.cur[\"attn\"].setdefault(i, dict(maxp=[], ent=[], map=None))\n        d[\"maxp\"].append(maxp)\n        d[\"ent\"].append(ent)\n        if k == 0:\n            d[\"map\"] = w[0].cpu().tolist()\n\n    def _cap_relu(self, i, h):\n        if not self.active:\n            return\n        h = h.detach().reshape(-1, h.shape[-1])\n        self.cur[\"relu\"][i] = dict(zero=(h <= 0).float().mean().item(),\n                                   dead=(~(h > 0).any(0)).float().mean().item())\n\n    def _on_opt_pre(self, opt, args, kwargs):\n        if not (self.active and self.cur and self.cur[\"loss\"] is not None):\n            return\n        self._snap, pg = {}, {}\n        for key, ps in self._groups:\n            ps = [p for p in ps if p.grad is not None]\n            self._snap[key] = [(p, p.detach().clone()) for p in ps if p.dim() >= 2]\n            if ps:\n                pg[key] = math.sqrt(sum(p.grad.float().pow(2).sum().item() for p in ps) / sum(p.numel() for p in ps))\n        self.cur[\"pgrad\"] = pg\n\n    def _on_opt_post(self, opt, args, kwargs):\n        if self.active and self.cur and self.cur[\"loss\"] is not None:\n            ur = {}\n            for key, pairs in self._snap.items():\n                vals = [_log(((p.detach() - old).std() / (old.std() + 1e-12)).item()) for p, old in pairs]\n                if vals:\n                    ur[key] = sum(vals) / len(vals)\n            self.cur[\"update\"], self.cur[\"step\"] = ur, self.steps\n            self._snap = {}\n            self._commit()\n        self.active = False\n        self.steps += 1\n\n    # ---- data --------------------------------------------------------------------\n\n    def _points(self):\n        \"\"\"Stream positions in order: s0 (embeddings), then mid/out of each block.\"\"\"\n        return [(\"s\", j) for j in range(2 * self.L + 1)]\n\n    def _commit(self):\n        c = self.cur\n        pts = [p for p in self._points() if p in c[\"fwd\"]]\n        ref = c[\"grad\"].get(pts[-1]) if pts else None\n        c[\"profile\"] = [(p[1], _log(c[\"grad\"][p] / ref)) for p in pts if ref and p in c[\"grad\"]]\n        self.hist.append(dict(step=c[\"step\"], loss=c[\"loss\"], profile=c[\"profile\"]))\n        FlowScope.runs[self.label] = dict(profile=c[\"profile\"], L=self.L, step=c[\"step\"], loss=c[\"loss\"])\n        self._save_runs()\n        if self._tour and len(c[\"cloud\"]) == self.L + 1:\n            clouds = [c[\"cloud\"][j] for j in range(self.L + 1)]\n            tokens = c[\"tokens\"] if c[\"tokens\"] and len(c[\"tokens\"]) == clouds[0].shape[0] else None\n            names = [\"emb\"] + [f\"b{i + 1}\" for i in range(self.L)]\n            self.tour_payload = self._tour.payload(clouds, names, tokens, c[\"T\"], c[\"step\"])\n            if self.record:\n                os.makedirs(os.path.dirname(os.path.abspath(self.record)), exist_ok=True)\n                with open(self.record, \"a\") as f:\n                    f.write(json.dumps(dict(self.tour_payload, label=self.label)) + \"\\n\")\n        self.render()\n\n    # runs are persisted in script mode so separate `python train.py` invocations can be compared\n    def _load_runs(self):\n        try:\n            with open(self.runs_file) as f:\n                for k, v in json.load(f).items():\n                    FlowScope.runs.setdefault(k, v)\n        except (OSError, ValueError):\n            pass\n        FlowScope.runs.pop(self.label, None)  # re-running the same config replaces its old line\n\n    def _save_runs(self):\n        if not getattr(self, \"runs_file\", None):\n            return\n        keep = dict(list(FlowScope.runs.items())[-6:])\n        os.makedirs(os.path.dirname(self.runs_file) or \".\", exist_ok=True)\n        with open(self.runs_file, \"w\") as f:\n            json.dump(keep, f)\n\n    # ---- rendering ---------------------------------------------------------------\n\n    def render(self):\n        html = self.to_html()\n        if self._handle is not None:\n            self._handle.update(self._HTML(html + (embed(self.tour_payload) if self.tour_payload else \"\")))\n        if self._server is not None:\n            self._server.publish(html, self.tour_payload)\n        return html\n\n    def save(self, path=\"flowscope.html\"):\n        with open(path, \"w\") as f:\n            f.write(\"<!doctype html><meta charset=utf-8><title>FlowScope</title>\"\n                    \"<body style='margin:0;background:#0b0e16;padding:8px'>\" + self.to_html()\n                    + (embed(self.tour_payload) if self.tour_payload else \"\") + \"</body>\")\n        return path\n\n    @staticmethod\n    def _frame(inner):\n        return ('<div style=\"background:#0b0e16;color:#d5dae6;font:12px ui-monospace,SFMono-Regular,Menlo,monospace;'\n                'border-radius:10px;padding:10px 12px;max-width:100%;overflow-x:auto\">' + inner + \"</div>\")\n\n    def to_html(self):\n        c = self.cur or {}\n        if not c.get(\"profile\") and not self.hist:\n            return self._frame(\"waiting for data\")\n        return self._frame(self._header(c) + self._diagram(c) + self._charts())\n\n    def _header(self, c):\n        prof = dict(c.get(\"profile\", []))\n        r = prof.get(0)\n        if r is None:\n            verdict, col = \"no gradient reached the embeddings\", \"#f87171\"\n        elif r < -3:\n            verdict, col = f\"BLOCKED: embeddings get 10^{r:.1f} of the output gradient\", \"#f87171\"\n        elif r < -1:\n            verdict, col = f\"weak: embeddings get 10^{r:.1f} of the output gradient\", \"#fbbf24\"\n        elif r > 2:\n            verdict, col = f\"EXPLODING: embeddings get 10^{r:+.1f} of the output gradient\", \"#e879f9\"\n        else:\n            verdict, col = f\"healthy: embeddings get {10 ** r:.2f}x of the output gradient\", \"#22d3ee\"\n        return (f'<div style=\"display:flex;gap:18px;flex-wrap:wrap;align-items:baseline;margin-bottom:4px\">'\n                f'<b style=\"font-size:14px;color:#fff\">FlowScope \u00b7 {self.label}</b>'\n                f'<span>step {c.get(\"step\", \"-\")}</span><span>loss {c.get(\"loss\", 0):.3f}</span>'\n                f'<span>{self.L} blocks</span><span style=\"color:{col}\">{verdict}</span></div>')\n\n    def _diagram(self, c):\n        L = self.L\n        dx = 128 if L <= 6 else max(64, 760 // L)\n        X0, W = 150, 150 + L * dx + 190\n        xs = {(\"s\", j): X0 + j * dx / 2 for j in range(2 * L + 1)}\n        xlog, xloss = xs[(\"s\", 2 * L)] + 80, xs[(\"s\", 2 * L)] + 150\n        YA, YF, YB, YS = 46, 116, 206, 238\n        out, anim = [], []\n        T = lambda x, y, s, fill=\"#8b93a7\", anchor=\"middle\", size=10: out.append(\n            f'<text x=\"{x:.0f}\" y=\"{y:.0f}\" fill=\"{fill}\" font-size=\"{size}\" text-anchor=\"{anchor}\">{s}</text>')\n        T(12, YF + 4, \"forward x\", \"#d5dae6\", \"start\", 11)\n        T(12, YF + 18, \"RMS\", \"#6b7280\", \"start\")\n        T(12, YB + 4, \"backward \u2207x\", \"#d5dae6\", \"start\", 11)\n        T(12, YB + 18, \"vs output\", \"#6b7280\", \"start\")\n\n        pts = self._points()\n        fwd, grad = c.get(\"fwd\", {}), c.get(\"grad\", {})\n        gref = grad.get(pts[-1])\n        glog = {p: (_log(grad[p] / gref) if gref and p in grad else None) for p in pts}\n        chain = [p for p in pts if p in fwd]\n        lane = [(xs[p], p) for p in chain] + [(xlog, (\"logits\", 0)), (xloss, \"loss\")]\n\n        # branch boxes (attention above, MLP below), drawn first so the lanes sit on top\n        for i in range(L):\n            for kind, x_add, ybox, name in ((\"a\", xs[(\"s\", 2 * i + 1)], YA, \"attn\"), (\"f\", xs[(\"s\", 2 * i + 2)], YF + 45, \"mlp\")):\n                key = (kind, i)\n                x_tap = x_add - dx / 2 + 10\n                bw = min(56, dx / 2 - 14)\n                bx = (x_tap + x_add) / 2\n                contrib = fwd.get(key, 0) / (fwd.get((\"s\", 2 * i + (0 if kind == \"a\" else 1)), 0) or 1)\n                ur = c.get(\"update\", {}).get(key)\n                # box outline = update/data ratio: cyan around 1e-3, red when the sublayer barely moves, magenta when too fast\n                edge = \"#374151\" if ur is None else _health(0 if -4 <= ur <= -2 else (-5 if ur < -4 else 2))\n                out.append(f'<path d=\"M{x_tap:.0f},{YF} L{x_tap:.0f},{ybox} L{x_add:.0f},{ybox} L{x_add:.0f},{YF}\" '\n                           f'fill=\"none\" stroke=\"#2a3142\" stroke-width=\"2\"/>')\n                out.append(f'<rect x=\"{bx - bw / 2:.0f}\" y=\"{ybox - 13}\" width=\"{bw:.0f}\" height=\"26\" rx=\"5\" '\n                           f'fill=\"#151a26\" stroke=\"{edge}\" stroke-width=\"1.5\"/>')\n                T(bx, ybox - 1, f\"{name}{i + 1}\" if dx >= 90 else name[0] + str(i + 1), \"#cbd5e1\")\n                if dx >= 90:\n                    T(bx, ybox + 9, f\"\u00d7{contrib:.2f}\", \"#6b7280\", size=8)\n                T(x_add, YF - 8 if kind == \"a\" else YF + 14, \"\u2295\", \"#4b5563\", size=11)\n\n        # lanes\n        for (xa, pa), (xb, pb) in zip(lane, lane[1:]):\n            fa = fwd.get(pa) if pa != \"loss\" else None\n            fcol = _fwd_color(fa) if fa else \"#374151\"\n            gb = glog.get(pa) if isinstance(pa, tuple) and pa[0] == \"s\" else (\n                _log(grad[pa] / gref) if gref and pa in grad else (0 if pa == (\"s\", 2 * L) else None))\n            gcol = _health(gb) if gb is not None else \"#374151\"\n            out.append(f'<line x1=\"{xa:.0f}\" y1=\"{YF}\" x2=\"{xb:.0f}\" y2=\"{YF}\" stroke=\"{fcol}\" stroke-width=\"7\" stroke-linecap=\"round\" opacity=\".6\"/>')\n            out.append(f'<line x1=\"{xa:.0f}\" y1=\"{YB}\" x2=\"{xb:.0f}\" y2=\"{YB}\" stroke=\"{gcol}\" stroke-width=\"7\" stroke-linecap=\"round\" opacity=\".6\"/>')\n\n        # nodes + numbers\n        for x, p in lane:\n            label = {\"logits\": \"logits\", \"loss\": \"loss\"}.get(p if p == \"loss\" else p[0])\n            if label is None:\n                j = p[1]\n                label = \"emb\" if j == 0 else (f\"b{(j + 1) // 2}\" if j % 2 == 0 else \"\")\n            out.append(f'<circle cx=\"{x:.0f}\" cy=\"{YF}\" r=\"4\" fill=\"#0b0e16\" stroke=\"#9ca3af\"/>')\n            out.append(f'<circle cx=\"{x:.0f}\" cy=\"{YB}\" r=\"4\" fill=\"#0b0e16\" stroke=\"#9ca3af\"/>')\n            if p == \"loss\":\n                T(x, YF - 12, \"loss\", \"#d5dae6\")\n                T(x, YF + 22, f\"{c.get('loss', 0):.2f}\", \"#fff\", size=11)\n                T(x, YB + 22, \"1.0\", \"#6b7280\")\n                continue\n            if label:\n                T(x, YF - 12 if p[0] != \"s\" or p[1] % 2 == 0 else YF - 12, label, \"#d5dae6\")\n            if p[0] == \"logits\" or p[1] % 2 == 0 or dx >= 110:\n                T(x, YF + 24 if p[0] == \"s\" and p[1] % 2 else YF + 24, _sci(fwd.get(p)), \"#9ca3af\", size=9)\n                gtxt = _sci(grad.get(p) / gref) if gref and grad.get(p) is not None else \"-\"\n                T(x, YB + 22, gtxt, \"#9ca3af\", size=9)\n\n        # pulses: forward sweep left to right, then the gradient sweeps back right to left\n        cyc, n = 7.0, len(lane) - 1\n        tf, tb0, tb = 0.40, 0.48, 0.40\n        for j in range(n):\n            (xa, pa), (xb, pb) = lane[j], lane[j + 1]\n            fa = fwd.get(pa) if pa != \"loss\" else None\n            r = 3 + 3 * min(1, max(0, (_log(fa) + 1) / 2)) if fa else 3\n            a, b = 0.005 + tf * j / n, 0.005 + tf * (j + 1) / n\n            anim.append(self._pulse(xa, xb, YF, r, _fwd_color(fa) if fa else \"#9ca3af\", a, b, cyc))\n            # backward: segment j is traversed from xb to xa, in reverse order\n            k = n - 1 - j\n            g = glog.get(pa) if isinstance(pa, tuple) and pa in glog else None\n            if g is None and pa == (\"logits\", 0):\n                g = 0\n            rb = max(1.2, 6 + g * 0.6) if g is not None else 1.2\n            a, b = tb0 + tb * k / n, tb0 + tb * (k + 1) / n\n            anim.append(self._pulse(xb, xa, YB, rb, _health(g) if g is not None else \"#7f1d1d\", a, b, cyc))\n\n        # vector strips: the actual numbers of one token's residual vector and its gradient\n        vec, gvec = c.get(\"vec\", {}), c.get(\"gvec\", {})\n        for p in chain:\n            if p[1] % 2 == 0 or dx >= 110:\n                x = xs[p]\n                if vec.get(p):\n                    out.append(f'<image href=\"{_strip(vec[p])}\" x=\"{x - 11:.0f}\" y=\"{YS}\" width=\"9\" height=\"96\" '\n                               f'preserveAspectRatio=\"none\" style=\"image-rendering:pixelated\"/>')\n                if gvec.get(p):\n                    out.append(f'<image href=\"{_strip(gvec[p])}\" x=\"{x + 2:.0f}\" y=\"{YS}\" width=\"9\" height=\"96\" '\n                               f'preserveAspectRatio=\"none\" style=\"image-rendering:pixelated\"/>')\n        T(12, YS + 44, \"one token:\", \"#6b7280\", \"start\")\n        T(12, YS + 58, \"x | \u2207x\", \"#6b7280\", \"start\")\n\n        # per-block internals: attention pattern (head 1) + saturation, MLP dead units, update ratio\n        YI = YS + 118\n        T(12, YI + 26, \"attn head 1\", \"#6b7280\", \"start\")\n        T(12, YI + 40, \"max-p \u00b7 dead\", \"#6b7280\", \"start\")\n        for i in range(L):\n            bx = (xs[(\"s\", 2 * i)] + xs[(\"s\", 2 * i + 2)]) / 2\n            att = c.get(\"attn\", {}).get(i)\n            sz = min(64, dx - 16)\n            if att and att[\"map\"]:\n                out.append(f'<image href=\"{_heat(att[\"map\"])}\" x=\"{bx - sz / 2:.0f}\" y=\"{YI}\" width=\"{sz:.0f}\" height=\"{sz:.0f}\" '\n                           f'style=\"image-rendering:pixelated\"/>')\n                mp = sum(att[\"maxp\"]) / len(att[\"maxp\"])\n                T(bx, YI + sz + 12, f\"p\u0302 {mp:.2f}\", \"#fca5a5\" if mp > 0.9 else \"#9ca3af\", size=9)\n            rl = c.get(\"relu\", {}).get(i)\n            if rl:\n                T(bx, YI + sz + 23, f\"dead {rl['dead'] * 100:.0f}%\", \"#fca5a5\" if rl[\"dead\"] > 0.2 else \"#9ca3af\", size=9)\n            ur = [c.get(\"update\", {}).get((k, i)) for k in \"af\"]\n            ur = [u for u in ur if u is not None]\n            if ur:\n                u = sum(ur) / len(ur)\n                T(bx, YI + sz + 34, f\"upd 10^{u:.1f}\", \"#fca5a5\" if u > -2 or u < -4.5 else \"#9ca3af\", size=9)\n        H = YI + 100\n        legend = (\"pulse size = signal size \u00b7 lane color: <span style='color:#22d3ee'>healthy</span> \"\n                  \"<span style='color:#9ca3af'>fading</span> <span style='color:#dc2626'>blocked</span> \"\n                  \"<span style='color:#e879f9'>exploding</span> \u00b7 p\u0302 = avg max attention prob (\u21921 = saturated softmax) \u00b7 \"\n                  \"dead = MLP units off for the whole batch \u00b7 upd = update/data ratio (~10^-3 is healthy; box outline too)\")\n        return (f'<div style=\"color:#6b7280;font-size:10px;margin:2px 0 6px\">{legend}</div>' f'<svg viewBox=\"0 0 {W:.0f} {H}\" width=\"{W:.0f}\" style=\"max-width:100%;height:auto;display:block\" '\n                f'xmlns=\"http://www.w3.org/2000/svg\">' + \"\".join(out) + \"\".join(anim) + \"</svg>\")\n\n    @staticmethod\n    def _pulse(x0, x1, y, r, col, a, b, cyc):\n        e = 0.004\n        return (f'<circle cx=\"{x0:.0f}\" cy=\"{y}\" r=\"{r:.1f}\" fill=\"#f8fafc\" stroke=\"{col}\" stroke-width=\"2\" opacity=\"0\" '\n                f'style=\"filter:drop-shadow(0 0 5px {col})\">'\n                f'<animate attributeName=\"cx\" values=\"{x0:.0f};{x0:.0f};{x1:.0f};{x1:.0f}\" keyTimes=\"0;{a:.3f};{b:.3f};1\" '\n                f'dur=\"{cyc}s\" repeatCount=\"indefinite\"/>'\n                f'<animate attributeName=\"opacity\" values=\"0;0;1;1;0;0\" '\n                f'keyTimes=\"0;{a:.3f};{a + e:.3f};{b - e:.3f};{b:.3f};1\" dur=\"{cyc}s\" repeatCount=\"indefinite\"/></circle>')\n\n    def _charts(self):\n        W, H, pad = 460, 190, 34\n        # left: gradient reaching each depth (log10, relative to the last block), this run over time + other runs\n        lo = -1.0\n        for h in self.hist[-12:]:\n            lo = min([lo] + [v for _, v in h[\"profile\"]])\n        others = {k: v for k, v in FlowScope.runs.items() if k != self.label and v[\"profile\"]}\n        for v in others.values():\n            lo = min([lo] + [y for _, y in v[\"profile\"]])\n        lo = math.floor(max(lo, -16))\n        hi = 1.0\n\n        def path(profile, L):\n            n = 2 * L\n            pts = [(pad + (W - pad - 10) * j / n, 10 + (H - 30) * (hi - max(y, lo)) / (hi - lo)) for j, y in profile]\n            return \"M\" + \" L\".join(f\"{x:.1f},{y:.1f}\" for x, y in pts) if pts else \"\"\n\n        g = [f'<text x=\"{pad}\" y=\"-2\" fill=\"#d5dae6\" font-size=\"11\">gradient reaching each depth (log10, vs output)</text>']\n        for t in range(int(lo), int(hi) + 1, max(1, int((hi - lo) // 6))):\n            y = 10 + (H - 30) * (hi - t) / (hi - lo)\n            g.append(f'<line x1=\"{pad}\" x2=\"{W - 10}\" y1=\"{y:.0f}\" y2=\"{y:.0f}\" stroke=\"#1f2533\"/>'\n                     f'<text x=\"{pad - 4}\" y=\"{y + 3:.0f}\" fill=\"#6b7280\" font-size=\"9\" text-anchor=\"end\">{t}</text>')\n        g.append(f'<text x=\"{pad}\" y=\"{H - 4}\" fill=\"#6b7280\" font-size=\"9\">emb</text>'\n                 f'<text x=\"{W - 10}\" y=\"{H - 4}\" fill=\"#6b7280\" font-size=\"9\" text-anchor=\"end\">last block</text>')\n        for k, (name, v) in enumerate(others.items()):\n            g.append(f'<path d=\"{path(v[\"profile\"], v[\"L\"])}\" fill=\"none\" stroke=\"#f59e0b\" stroke-width=\"1.5\" '\n                     f'stroke-dasharray=\"4 3\" opacity=\".8\"/>')\n        recent = self.hist[-12:]\n        for k, h in enumerate(recent):\n            last = k == len(recent) - 1\n            g.append(f'<path d=\"{path(h[\"profile\"], self.L)}\" fill=\"none\" stroke=\"#22d3ee\" '\n                     f'stroke-width=\"{2.2 if last else 1}\" opacity=\"{1 if last else 0.12 + 0.5 * k / len(recent)}\"/>')\n\n        # right: loss\n        ls = [(h[\"step\"], h[\"loss\"]) for h in self.hist]\n        s0, s1 = ls[0][0], max(ls[-1][0], ls[0][0] + 1)\n        l0, l1 = min(l for _, l in ls), max(l for _, l in ls)\n        l1 = l1 if l1 > l0 else l0 + 1\n        lp = \"M\" + \" L\".join(f\"{pad + (W - pad - 10) * (s - s0) / (s1 - s0):.1f},{10 + (H - 30) * (l1 - l) / (l1 - l0):.1f}\"\n                             for s, l in ls)\n        lchart = [f'<text x=\"{pad}\" y=\"-2\" fill=\"#d5dae6\" font-size=\"11\">training loss (sampled)</text>',\n                  f'<text x=\"{pad - 4}\" y=\"13\" fill=\"#6b7280\" font-size=\"9\" text-anchor=\"end\">{l1:.2f}</text>',\n                  f'<text x=\"{pad - 4}\" y=\"{H - 20}\" fill=\"#6b7280\" font-size=\"9\" text-anchor=\"end\">{l0:.2f}</text>',\n                  f'<text x=\"{W - 10}\" y=\"{H - 4}\" fill=\"#6b7280\" font-size=\"9\" text-anchor=\"end\">step {s1}</text>',\n                  f'<path d=\"{lp}\" fill=\"none\" stroke=\"#a78bfa\" stroke-width=\"1.6\"/>']\n        svg = lambda body: (f'<svg viewBox=\"0 -14 {W} {H + 14}\" width=\"{W}\" style=\"max-width:100%;height:auto\" '\n                            f'xmlns=\"http://www.w3.org/2000/svg\">{\"\".join(body)}</svg>')\n        key = [f'<div style=\"color:#22d3ee\">\u2501 {self.label} (faded = earlier steps)</div>']\n        key += [f'<div style=\"color:#f59e0b\">\u254d {n} ({v[\"L\"]} blocks, step {v[\"step\"]}, loss {v[\"loss\"]:.2f})</div>'\n                for n, v in others.items()]\n        return (f'<div style=\"display:flex;gap:16px;flex-wrap:wrap;margin-top:8px\">'\n                f'<div>{svg(g)}<div style=\"font-size:10px;margin-left:{pad}px\">{\"\".join(key)}</div></div>'\n                f'<div>{svg(lchart)}</div></div>')\n",
"flowscope/server.py": "\"\"\"Tiny live page: the training process publishes HTML frames, browsers get them over Server-Sent Events.\n\nStdlib only. The page keeps the last frame when the script exits and reconnects on its own when you\nrun the next script, so you can leave the tab open while you iterate.\n\"\"\"\nimport json, threading\n\nfrom flowscope.tour import JS as TOUR_JS\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\nPAGE = \"\"\"<!doctype html><html><head><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">\n<title>FlowScope</title>\n<style>\n  body{margin:0;background:#0b0e16;color:#d5dae6;font:12px ui-monospace,SFMono-Regular,Menlo,monospace}\n  #bar{display:flex;gap:10px;align-items:center;padding:6px 14px;color:#6b7280;font-size:11px}\n  #dot{width:8px;height:8px;border-radius:50%;background:#6b7280}\n  #root{padding:0 8px 16px}\n</style></head><body>\n<div id=\"bar\"><span id=\"dot\"></span><span id=\"status\">connecting...</span></div>\n<div id=\"root\"></div>\n<div id=\"tour\" style=\"padding:0 14px 24px\"></div>\n<script>%TOUR_JS%</script>\n<script>\n  const root = document.getElementById('root'), dot = document.getElementById('dot'), status = document.getElementById('status');\n  const set = (c, t) => { dot.style.background = c; status.textContent = t; };\n  const es = new EventSource('/events');\n  es.onopen = () => set('#22d3ee', 'live');\n  es.onmessage = e => {\n    const m = JSON.parse(e.data);\n    root.innerHTML = m.html;\n    if (m.tour) FlowTour.attach(document.getElementById('tour'), m.tour);\n    set('#22d3ee', 'live \u00b7 updated ' + new Date().toLocaleTimeString());\n  };\n  es.onerror = () => set('#6b7280', 'run ended or not started \u00b7 showing last frame \u00b7 reconnects automatically');\n</script></body></html>\"\"\"\nPAGE = PAGE.replace(\"%TOUR_JS%\", TOUR_JS.replace(\"</\", \"<\\\\/\"))\n\n\ndef _lan_ip():\n    \"\"\"The address other devices on the LAN use to reach this machine (no packets are sent).\"\"\"\n    import socket\n    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:\n        try:\n            s.connect((\"10.255.255.255\", 1))\n            return s.getsockname()[0]\n        except OSError:\n            return \"127.0.0.1\"\n\n\nclass LiveServer:\n    _shared = {}\n\n    @classmethod\n    def shared(cls, port=8765, host=\"127.0.0.1\"):\n        \"\"\"One server per process and port, so several scopes (or reruns) reuse it.\"\"\"\n        if port not in cls._shared:\n            cls._shared[port] = cls(port, host)\n        return cls._shared[port]\n\n    def __init__(self, port=8765, host=\"127.0.0.1\"):\n        self.frame, self.tour, self.version, self.cond = \"\", None, 0, threading.Condition()\n        server = self\n\n        class Handler(BaseHTTPRequestHandler):\n            def log_message(self, *a):\n                pass\n\n            def do_GET(self):\n                if self.path == \"/events\":\n                    return self._events()\n                body = (PAGE if self.path in (\"/\", \"/index.html\") else server.frame).encode()\n                self.send_response(200)\n                self.send_header(\"Content-Type\", \"text/html; charset=utf-8\")\n                self.send_header(\"Content-Length\", str(len(body)))\n                self.end_headers()\n                self.wfile.write(body)\n\n            def _events(self):\n                self.send_response(200)\n                self.send_header(\"Content-Type\", \"text/event-stream\")\n                self.send_header(\"Cache-Control\", \"no-cache\")\n                self.end_headers()\n                seen = -1\n                try:\n                    while True:\n                        with server.cond:\n                            server.cond.wait_for(lambda: server.version != seen, timeout=15)\n                            frame, tour, v = server.frame, server.tour, server.version\n                        msg = f\"data: {json.dumps(dict(html=frame, tour=tour))}\\n\\n\" if v != seen else \": keepalive\\n\\n\"\n                        seen = v\n                        self.wfile.write(msg.encode())\n                        self.wfile.flush()\n                except (BrokenPipeError, ConnectionResetError):\n                    pass\n\n        for p in range(port, port + 20):  # next free port if something else holds this one\n            try:\n                self.httpd = ThreadingHTTPServer((host, p), Handler)\n                break\n            except OSError:\n                continue\n        else:\n            raise OSError(f\"FlowScope: no free port in {port}-{port + 19}\")\n        self.httpd.daemon_threads = True\n        port = self.httpd.server_address[1]\n        self.url = f\"http://{host}:{port}\"\n        threading.Thread(target=self.httpd.serve_forever, daemon=True).start()\n        print(f\"FlowScope live at {self.url}\", flush=True)\n        if host in (\"0.0.0.0\", \"\"):\n            print(f\"FlowScope: reachable from your network (e.g. Vision Pro) at http://{_lan_ip()}:{port}\", flush=True)\n\n    def publish(self, html, tour=None):\n        with self.cond:\n            self.frame, self.tour, self.version = html, tour, self.version + 1\n            self.cond.notify_all()\n",
"flowscope/tour.py": "\"\"\"Representation tour: the residual-stream point cloud at every block, packed for a grand tour in the browser.\n\nAll blocks are expressed in one shared principal basis of the residual stream, so the same 3D window\nlooks at every block at the same moment and you can watch the cloud get reorganized with depth.\n\"\"\"\nimport base64, json, os\n\nimport torch\n\nJS = open(os.path.join(os.path.dirname(os.path.abspath(__file__)), \"tour.js\")).read()\n\n\ndef sample(t, n_points):\n    \"\"\"The first few whole sequences of a (B, T, C) activation, as (N, C) float32 on the cpu.\"\"\"\n    B, T, C = t.shape\n    return t.detach()[:rows(B, T, n_points)].reshape(-1, C).float().cpu()\n\n\ndef rows(B, T, n_points):\n    return max(1, min(B, n_points // T))\n\n\ndef effective_dim(X):\n    \"\"\"Participation ratio of the covariance spectrum: ~C for an isotropic cloud, ~1 for a line.\"\"\"\n    s = torch.linalg.svdvals(X - X.mean(0)) ** 2\n    return float(s.sum() ** 2 / (s ** 2).sum()) if s.sum() > 0 else 0.0\n\n\ndef _procrustes(A, ref):\n    \"\"\"Rotate A's columns within their own span to best match ref, so the view doesn't jump between updates.\"\"\"\n    U, _, Vh = torch.linalg.svd(A.T @ ref)\n    return A @ (U @ Vh)\n\n\nclass Tour:\n    def __init__(self, k=8):\n        self.k, self.W = k, None\n\n    def payload(self, clouds, names, tokens, T, step):\n        Xs = []\n        for X in clouds:\n            X = X - X.mean(0)\n            Xs.append(X / X.pow(2).mean().sqrt().clamp_min(1e-12))  # shape only; magnitude is in the lanes above\n        W = torch.linalg.svd(torch.cat(Xs), full_matrices=False).Vh.T  # C x C, columns = principal directions\n        k = min(self.k, W.shape[1])\n        if self.W is not None and self.W.shape == W.shape:\n            W = torch.cat([_procrustes(W[:, :k], self.W[:, :k]), _procrustes(W[:, k:], self.W[:, k:])], 1)\n        self.W = W\n        layers = []\n        for name, X in zip(names, Xs):\n            Z = X @ W\n            m = float(Z.abs().max().clamp_min(1e-12))\n            q = (Z / m * 127).round().to(torch.int8).view(torch.uint8).flatten().tolist()\n            layers.append(dict(name=name, dim=round(effective_dim(X), 1), scale=m / 127,\n                               data=base64.b64encode(bytes(q)).decode()))\n        return dict(step=step, C=W.shape[0], N=Xs[0].shape[0], T=T, k=k, layers=layers, tokens=tokens)\n\n\ndef embed(payload):\n    \"\"\"Self-contained HTML for notebooks: the engine plus this frame's data.\"\"\"\n    data = json.dumps(payload).replace(\"</\", \"<\\\\/\")\n    return (f'<div class=\"fs-tour-slot\"></div><script>{JS}\\n'\n            f'FlowTour.attach(document.currentScript ? document.currentScript.previousElementSibling : '\n            f'[...document.querySelectorAll(\".fs-tour-slot\")].pop(), {data});</script>')\n",
"flowscope/tour.js": "// FlowScope representation tour: a grand tour (Asimov's torus method) through the residual stream.\n// Every panel is one block's token cloud; all panels share the same moving 3D window.\n(function () {\n  if (window.FlowTour) return;\n\n  const CSS = `\n  .fs-tour{margin-top:14px;color:#d5dae6;font:12px ui-monospace,SFMono-Regular,Menlo,monospace}\n  .fs-tour-head{display:flex;gap:14px;align-items:baseline;flex-wrap:wrap;margin-bottom:6px}\n  .fs-tour-head b{font-size:13px;color:#fff}\n  .fs-tour-ctl{display:flex;gap:16px;flex-wrap:wrap;align-items:center;color:#8b93a7;margin-bottom:8px}\n  .fs-tour-ctl input[type=range]{vertical-align:middle;width:110px;accent-color:#22d3ee}\n  .fs-tour-ctl button,.fs-tour-ctl select{background:#151a26;color:#d5dae6;border:1px solid #2a3142;border-radius:5px;\n    padding:3px 8px;font:inherit;cursor:pointer}\n  .fs-tour-grid{display:flex;flex-wrap:wrap;gap:8px}\n  .fs-panel{background:#0f1320;border:1px solid #1f2533;border-radius:8px;padding:4px;cursor:zoom-in}\n  .fs-panel.big{cursor:zoom-out;border-color:#2f3a52}\n  .fs-panel .lbl{display:flex;justify-content:space-between;color:#8b93a7;font-size:10px;padding:0 3px 2px}\n  .fs-panel .lbl b{color:#d5dae6;font-weight:600}\n  .fs-panel canvas{display:block}\n  .fs-tour-note{color:#6b7280;font-size:10px;margin-top:6px;max-width:900px;line-height:1.5}`;\n\n  const VIRIDIS = [[68, 1, 84], [59, 82, 139], [33, 145, 140], [94, 201, 98], [253, 231, 37]];\n  const SMALL = 168, BIG = 420;\n\n  function viridis(t) {\n    t = Math.min(1, Math.max(0, t)) * (VIRIDIS.length - 1);\n    const i = Math.min(VIRIDIS.length - 2, Math.floor(t)), f = t - i, a = VIRIDIS[i], b = VIRIDIS[i + 1];\n    return `rgb(${a.map((v, k) => Math.round(v + (b[k] - v) * f)).join(\",\")})`;\n  }\n\n  function decode(layer, N, C) {\n    const bin = atob(layer.data), z = new Float32Array(N * C);\n    for (let i = 0; i < z.length; i++) {\n      const b = bin.charCodeAt(i);\n      z[i] = (b > 127 ? b - 256 : b) * layer.scale;\n    }\n    const v = new Float64Array(C);  // variance along each principal direction (data is centered)\n    for (let i = 0; i < z.length; i++) v[i % C] += z[i] * z[i] / N;\n    return { name: layer.name, dim: layer.dim, z, v, total: v.reduce((a, b) => a + b, 0) || 1 };\n  }\n\n  // Torus method: rotate in every coordinate plane (i, j) at its own speed; with rationally independent\n  // speeds the path never repeats and comes arbitrarily close to every 3D projection.\n  function speeds(n) {\n    const out = [];\n    let norm = 0, m = 2;\n    for (let i = 0; i < n; i++)\n      for (let j = i + 1; j < n; j++) {\n        while (Number.isInteger(Math.sqrt(m))) m++;\n        const v = 0.25 + 0.75 * (Math.sqrt(m) % 1);  // fractional parts of sqrt(non-squares): irrational\n        out.push([i, j, v]);\n        norm += v * v;\n        m++;\n      }\n    const k = 0.5 * Math.sqrt(3) / Math.sqrt(norm);  // keep the visible motion calm whatever n is\n    return out.map(([i, j, v]) => [i, j, v * k * Math.sqrt(n)]);\n  }\n\n  class Engine {\n    constructor() {\n      if (!document.getElementById(\"fs-tour-css\")) {\n        const st = document.createElement(\"style\");\n        st.id = \"fs-tour-css\";\n        st.textContent = CSS;\n        document.head.appendChild(st);\n      }\n      this.el = document.createElement(\"div\");\n      this.el.className = \"fs-tour\";\n      this.el.innerHTML = `\n        <div class=\"fs-tour-head\"><b>representation tour</b><span class=\"info\"></span></div>\n        <div class=\"fs-tour-ctl\">\n          <label>dims toured <input class=\"n\" type=\"range\" min=\"3\" max=\"64\" value=\"8\"> <span class=\"nv\">8</span></label>\n          <label>speed <input class=\"sp\" type=\"range\" min=\"0\" max=\"3\" step=\"0.1\" value=\"1\"></label>\n          <button class=\"pause\">pause</button><button class=\"reset\">back to PC1-3</button>\n          <label>color <select class=\"col\"><option value=\"position\">position in sequence</option>\n            <option value=\"token\">token id</option></select></label>\n        </div>\n        <div class=\"fs-tour-grid\"></div>\n        <div class=\"fs-tour-note\">Each dot is one token's residual vector at that block, normalized to unit RMS (shape only;\n          size is in the lanes above). All panels look through the same moving 3D window, touring the top principal\n          directions of the residual stream. <b>eff.dim</b> = how many directions the cloud really uses (participation\n          ratio). Low at step 0 = rank collapse. <b>% seen</b> = share of the block's variance inside the toured\n          dims; when it is low, raise \"dims toured\" or the panel is mostly showing you what's left over. Click a panel to enlarge.</div>`;\n      const q = s => this.el.querySelector(s);\n      this.grid = q(\".fs-tour-grid\");\n      this.info = q(\".info\");\n      this.t = 0; this.speed = 1; this.paused = false; this.n = null; this.color = \"position\";\n      this.layers = []; this.prev = []; this.fadeStart = 0; this.panels = [];\n      q(\".n\").oninput = e => { this.n = +e.target.value; q(\".nv\").textContent = this.n; this.sp = speeds(this.n); this.label(); };\n      q(\".sp\").oninput = e => { this.speed = +e.target.value; };\n      q(\".pause\").onclick = e => { this.paused = !this.paused; e.target.textContent = this.paused ? \"play\" : \"pause\"; };\n      q(\".reset\").onclick = () => { this.t = 0; };\n      q(\".col\").onchange = e => { this.color = e.target.value; this.paint(); };\n      this.last = performance.now();\n      requestAnimationFrame(this.tick.bind(this));\n    }\n\n    update(p) {\n      if (!p || p.C < 3) return;\n      const first = this.n === null;\n      this.p = p;\n      this.prev = this.layers;\n      this.layers = p.layers.map(l => decode(l, p.N, p.C));\n      this.fadeStart = performance.now();\n      const nIn = this.el.querySelector(\".n\");\n      nIn.max = p.C;\n      if (first || this.n > p.C) {\n        this.n = Math.min(p.k, p.C);\n        nIn.value = this.n;\n        this.el.querySelector(\".nv\").textContent = this.n;\n        this.sp = speeds(this.n);\n      }\n      this.el.querySelector(\".col\").disabled = !p.tokens;\n      if (!p.tokens) this.color = \"position\";\n      this.info.textContent = `step ${p.step} \u00b7 ${p.N} tokens \u00d7 ${p.C} dims \u00b7 ${this.layers.length} panels`;\n      if (this.panels.length !== this.layers.length) this.buildPanels();\n      this.label();\n      this.paint();\n    }\n\n    // Zoom: a typical 3D view of an n-dim cloud carries ~3/n of its variance, so size panels by that, but cap it\n    // so a collapsed cloud still fits when the window lines up with it. Also report how much of the block is toured.\n    zoom(l) {\n      const { N, C } = this.p, n = this.n;\n      let inView = 0;\n      for (let d = 0; d < n; d++) inView += l.v[d];\n      const r = new Float64Array(N);\n      for (let p = 0; p < N; p++)\n        for (let d = 0; d < n; d++) r[p] += l.z[p * C + d] ** 2;\n      const r95 = Math.sqrt(r.sort()[Math.floor(0.95 * (N - 1))]);  // a few outlier tokens may leave the panel\n      l.fit = 1 / Math.max(2.6 * Math.sqrt(inView / n), r95 / 1.1, 1e-9);  // view units -> half-panel\n      return inView;\n    }\n\n    label() {\n      this.prev.forEach(l => this.zoom(l));\n      this.layers.forEach((l, i) => {\n        const pct = Math.round(100 * this.zoom(l) / l.total);\n        this.panels[i].lbl.innerHTML = `<b>${l.name}</b><span title=\"eff.dim: directions the cloud really uses \u00b7 `\n          + `% seen: share of its variance inside the toured dims\">dim ${l.dim} \u00b7 <span style=\"color:${\n          pct < 50 ? \"#fbbf24\" : \"inherit\"}\">${pct}% seen</span></span>`;\n      });\n    }\n\n    paint() {\n      const p = this.p;\n      if (!p) return;\n      this.colors = Array.from({ length: p.N }, (_, i) => this.color === \"token\" && p.tokens\n        ? `hsl(${(p.tokens[i] * 137.508) % 360},70%,62%)`\n        : viridis((i % p.T) / Math.max(1, p.T - 1)));\n    }\n\n    buildPanels() {\n      this.grid.innerHTML = \"\";\n      this.panels = this.layers.map(() => {\n        const div = document.createElement(\"div\");\n        div.className = \"fs-panel\";\n        const lbl = document.createElement(\"div\");\n        lbl.className = \"lbl\";\n        const cv = document.createElement(\"canvas\");\n        div.append(lbl, cv);\n        const panel = { div, lbl, cv, big: false };\n        div.onclick = () => { panel.big = !panel.big; div.classList.toggle(\"big\", panel.big); this.size(panel); };\n        this.size(panel);\n        this.grid.appendChild(div);\n        return panel;\n      });\n    }\n\n    size(panel) {\n      const s = panel.big ? BIG : SMALL, dpr = window.devicePixelRatio || 1;\n      panel.cv.width = panel.cv.height = s * dpr;\n      panel.cv.style.width = panel.cv.style.height = s + \"px\";\n      panel.px = s;\n    }\n\n    basis() {\n      const n = this.n, v = [new Float64Array(n), new Float64Array(n), new Float64Array(n)];\n      v[0][0] = v[1][1] = v[2][2] = 1;\n      for (const [i, j, w] of this.sp) {\n        const th = w * this.t, c = Math.cos(th), s = Math.sin(th);\n        for (const b of v) {\n          const a = b[i], d = b[j];\n          b[i] = c * a - s * d;\n          b[j] = s * a + c * d;\n        }\n      }\n      return v;\n    }\n\n    tick(now) {\n      const dt = Math.min(0.1, (now - this.last) / 1000);\n      this.last = now;\n      if (!this.paused) this.t += dt * this.speed;\n      if (this.layers.length && this.el.isConnected) {\n        const B = this.basis(), f = Math.min(1, (now - this.fadeStart) / 700);\n        this.panels.forEach((panel, i) => {\n          const ctx = panel.cv.getContext(\"2d\"), dpr = window.devicePixelRatio || 1;\n          ctx.setTransform(dpr, 0, 0, dpr, 0, 0);\n          ctx.clearRect(0, 0, panel.px, panel.px);\n          if (panel.big) this.spokes(ctx, panel.px, B);\n          if (f < 1 && this.prev[i]) this.draw(ctx, panel.px, this.prev[i], B, 1 - f);\n          this.draw(ctx, panel.px, this.layers[i], B, f);\n        });\n      }\n      requestAnimationFrame(this.tick.bind(this));\n    }\n\n    draw(ctx, S, layer, B, alpha) {\n      const { N, C } = this.p, n = this.n, z = layer.z, R = S * 0.44 * layer.fit;\n      const pts = new Array(N);\n      for (let p = 0; p < N; p++) {\n        let x = 0, y = 0, w = 0;\n        const o = p * C;\n        for (let d = 0; d < n; d++) {\n          const v = z[o + d];\n          x += B[0][d] * v; y += B[1][d] * v; w += B[2][d] * v;\n        }\n        pts[p] = [x, y, w * layer.fit, p];\n      }\n      pts.sort((a, b) => a[2] - b[2]);  // far points first\n      for (const [x, y, w, p] of pts) {\n        const depth = Math.min(1, Math.max(0, (w + 1) / 2)), k = 4 / (4 - Math.max(-1.5, Math.min(1.5, w)));\n        ctx.globalAlpha = alpha * (0.35 + 0.65 * depth);\n        ctx.fillStyle = this.colors[p];\n        ctx.beginPath();\n        ctx.arc(S / 2 + x * R * k, S / 2 - y * R * k, (1.1 + 1.6 * depth) * (S > SMALL ? 1.5 : 1), 0, 6.2832);\n        ctx.fill();\n      }\n      ctx.globalAlpha = 1;\n    }\n\n    spokes(ctx, S, B) {\n      // where each principal direction points in the current view (length = how much of it you are seeing)\n      ctx.font = \"10px ui-monospace,Menlo,monospace\";\n      for (let d = 0; d < Math.min(this.n, 8); d++) {\n        const x = S / 2 + B[0][d] * S * 0.42, y = S / 2 - B[1][d] * S * 0.42;\n        ctx.strokeStyle = ctx.fillStyle = \"rgba(139,147,167,0.45)\";\n        ctx.beginPath(); ctx.moveTo(S / 2, S / 2); ctx.lineTo(x, y); ctx.stroke();\n        ctx.fillText(\"PC\" + (d + 1), x + 3, y - 3);\n      }\n    }\n  }\n\n  window.FlowTour = {\n    attach(slot, payload) {\n      if (!window.__fsTour) window.__fsTour = new Engine();\n      const eng = window.__fsTour;\n      if (slot && eng.el.parentNode !== slot) slot.appendChild(eng.el);\n      eng.update(payload);\n    },\n  };\n})();\n"
}
for path, src in files.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        f.write(src)
for m in [m for m in sys.modules if m == 'flowscope' or m.startswith('flowscope.')]:
    del sys.modules[m]  # pick up the new files on re-run
print('flowscope written:', ', '.join(files))


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 2000 # 5000 in the original; 2000 is plenty to see the effects
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# --- FlowScope experiment knobs ---
use_residual = True     # x = x + f(x)  vs  x = f(x)
use_layernorm = True    # False swaps LayerNorms for identity
scale_attention = True  # the C**-0.5 in attention
run_label = f"{n_layer} layers, residual={use_residual}, ln={use_layernorm}, scaled={scale_attention}"
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * (C**-0.5 if scale_attention else 1.0) # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd) if use_layernorm else nn.Identity()
        self.ln2 = nn.LayerNorm(n_embd) if use_layernorm else nn.Identity()

    def forward(self, x):
        if use_residual:
            x = x + self.sa(self.ln1(x))
            x = x + self.ffwd(self.ln2(x))
        else:
            x = self.sa(self.ln1(x))
            x = self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

from flowscope import FlowScope
scope = FlowScope(model, optimizer, every=50, label=run_label)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


### Experiments to try (change the knobs at the top of the cell and re-run)

1. **Baseline**: defaults. Gradient reaches the embeddings at ~1-3x of the output. Healthy.
2. **Go deep, remove the residuals**: `n_layer = 12`, `use_residual = False`. The bottom lane fades to red toward the left (~1e-8 of the gradient reaches the embeddings) and gets worse as training goes on; loss stalls around 3.3-3.4 instead of dropping under 2.5, and most MLP units die.
3. **Also remove LayerNorm**: `use_layernorm = False`. Now it's blocked from step 0 (~1e-11 of the gradient reaches the embeddings), the forward signal also shrinks layer by layer, and half the MLP units die.
4. **Residuals back, still 12 layers, no LayerNorm**: `use_residual = True`, `use_layernorm = False`. The gradient gets through again (the "gradient highway") and loss beats the 4-layer baseline, but now the forward lane shows the stream growing ~10x with depth and attention saturating (`p̂` ~0.9). That's the job LayerNorm does.
5. **Unscaled attention**: `scale_attention = False`. `p̂` goes from ~0.25 to ~0.65: the softmax gets peakier (the "Scaled attention" note above). Mild here because head_size is only 16; it gets worse with bigger heads.
6. **Too-high learning rate**: `learning_rate = 3e-2`. Update/data ratios go red, the stream RMS explodes into the hundreds, gradients turn magenta, attention saturates and MLP units die.

`FlowScope.runs.clear()` wipes the saved comparison lines.
